# Configuración e importación de un modelo de robot desde Onshape hacia MuJoCo

Este notebook establece el flujo de trabajo completo para:
1. Instalar las librerías necesarias.
2. Configurar las credenciales de la API de Onshape.
3. Definir un archivo de configuración (`config.json`) avanzado que especifica:
   - URL del ensamblaje en Onshape.
   - Propiedades dinámicas de las articulaciones (actuación, límites, ganancias PID, fricción, etc.).
   - Formato de salida (MuJoCo).
   - Opciones de mallas visuales y de colisión.
4. Generar automáticamente el modelo MJCF mediante `onshape-to-robot`.
5. Visualizar y simular el robot en un entorno interactivo de MuJoCo, con control de parámetros.

El procedimiento sigue prácticas de ingeniería de software: verificación de dependencias, manejo de excepciones y cierre adecuado de recursos.

## 1. Instalación de la librería `onshape-to-robot`

Esta herramienta permite extraer ensamblajes desde Onshape y convertirlos a formatos compatibles con motores de simulación.  
Documentación oficial: [onshape-to-robot GitHub](https://github.com/RobotecAI/onshape-to-robot)

Se instala silenciosamente y se verifica la versión.

In [1]:
import sys
import subprocess

# Instalar onshape-to-robot
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onshape-to-robot", "-q"])
    print("Onshape-to-robot instalado correctamente.")
except subprocess.CalledProcessError as e:
    print(f"Error durante la instalación: {e}")
    sys.exit(1)

from importlib.metadata import version
print(f"Versión de onshape-to-robot: {version('onshape-to-robot')}")

Onshape-to-robot instalado correctamente.
Versión de onshape-to-robot: 1.8.2


## 2. Configuración de claves de acceso a la API de Onshape

Onshape requiere autenticación mediante `access_key` y `secret_key`.  
Se obtienen desde el [Portal de Desarrollador de Onshape](https://dev-portal.onshape.com/keys).  
Se configuran como variables de entorno.

In [2]:
import os
from pathlib import Path

# Cargar variables de entorno desde archivo .env (opcional)
env_file = Path(".env")
if env_file.exists():
    from dotenv import load_dotenv
    load_dotenv(env_file)
else:
    print("⚠️  Archivo .env no encontrado. Asegúrate de configurar las variables de entorno.")
    print("   Copia .env.example a .env y actualiza con tus credenciales reales.")

# Obtener credenciales desde variables de entorno
ONSHAPE_API = os.getenv('ONSHAPE_API', 'https://cad.onshape.com')
ONSHAPE_ACCESS_KEY = os.getenv('ONSHAPE_ACCESS_KEY')
ONSHAPE_SECRET_KEY = os.getenv('ONSHAPE_SECRET_KEY')

# Validar que las credenciales estén configuradas
if not ONSHAPE_ACCESS_KEY or not ONSHAPE_SECRET_KEY:
    raise RuntimeError(
        "❌ Credenciales de Onshape no configuradas.\n"
        "   1. Copia .env.example a .env\n"
        "   2. Obtén tus claves desde: https://dev-portal.onshape.com/keys\n"
        "   3. Actualiza ONSHAPE_ACCESS_KEY y ONSHAPE_SECRET_KEY en .env"
    )

# Configurar variables de entorno para la herramienta
os.environ['ONSHAPE_API'] = ONSHAPE_API
os.environ['ONSHAPE_ACCESS_KEY'] = ONSHAPE_ACCESS_KEY
os.environ['ONSHAPE_SECRET_KEY'] = ONSHAPE_SECRET_KEY


print("✅ Credenciales de Onshape configuradas desde variables de entorno.")

✅ Credenciales de Onshape configuradas desde variables de entorno.


## 3. Configuración avanzada del proyecto: `model/config.json`

El archivo de configuración permite controlar aspectos finos de la generación del modelo y la simulación.  
**Parámetros incluidos:**

- `url`: URL del ensamblaje en Onshape.
- `output_format`: Formato de salida (`mujoco`).
- `joint_propieties`: (Nótese que la clave correcta según la documentación es `joint_properties`; se usará la correcta internamente).  
  Para todas las articulaciones (`"*"`) se definen:
  - `actuated`: si el joint tiene actuador.
  - `forcerange`: par máximo (N·m).
  - `limits`: [mínimo, máximo] en radianes.
  - `frictionloss`: fricción viscosa.
  - `kv`: ganancia de velocidad.
  - `kp`: ganancia proporcional (posición).
- `output_filename`: nombre base del archivo de salida (sin extensión).
- `use_visual_meshes`: usar mallas visuales.
- `use_collision_meshes`: usar mallas de colisión.
- `simplify_collision_meshes`: simplificar geometrías de colisión.

> **Nota**: La clave `joint_propieties` se ha renombrado a `joint_properties` en el código para cumplir con el estándar de `onshape-to-robot`.

In [3]:
import json
import os
import sys

# URL del ensamblaje proporcionada
ASSEMBLY_URL = "https://cad.onshape.com/documents/d578f2fbdc332270e30f24e5/w/ba84a5b38e318fd89e8c140d/e/9356b596fb5246da0639ba81"

# Configuración completa según especificación, usando la clave correcta "joint_properties"
config = {
    "url": ASSEMBLY_URL,
    "output_format": "mujoco",
    "output_filename": "Robot",
    "use_visual_meshes": True,
    "use_collision_meshes": True,
    "simplify_collision_meshes": True
}

# Crear directorio 'model'
os.makedirs("model", exist_ok=True)

config_path = os.path.join("model", "config.json")
try:
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)
    print(f"Archivo de configuración enriquecido creado en: {config_path}")
    print("   Parámetros configurados:")
    for key, value in config.items():
        print(f"     - {key}: {value}")
except IOError as e:
    print(f"Error al escribir el archivo: {e}")
    sys.exit(1)

Archivo de configuración enriquecido creado en: model\config.json
   Parámetros configurados:
     - url: https://cad.onshape.com/documents/d578f2fbdc332270e30f24e5/w/ba84a5b38e318fd89e8c140d/e/9356b596fb5246da0639ba81
     - output_format: mujoco
     - output_filename: Robot
     - use_visual_meshes: True
     - use_collision_meshes: True
     - simplify_collision_meshes: True


## 4. Importación del robot desde Onshape

Se ejecuta `onshape-to-robot` sobre el directorio `model`.  
La herramienta leerá `config.json` y generará los archivos:
- `HexaBot.xml` (modelo principal)
- `scene.xml` (escena con suelo y luces)
- Mallas STL en `model/assets/`

Si alguna articulación tiene propiedades específicas, serán aplicadas en el archivo XML.

In [4]:
import subprocess
import sys

try:
    result = subprocess.run(
        ["onshape-to-robot", "model"],
        capture_output=True,
        text=True,
        check=False
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("Falló la generación del modelo:")
        print(result.stderr)
        sys.exit(result.returncode)
    else:
        print("Modelo generado exitosamente con la configuración enriquecida.")
except FileNotFoundError:
    print("El comando 'onshape-to-robot' no se encuentra.")
    sys.exit(1)
except Exception as e:
    print(f"Error inesperado: {e}")
    sys.exit(1)

* onshape-to-robot version 1.8.2
* Using configuration workspace ID ba84a5b38e318fd89e8c140d ...
* Using configuration element ID 9356b596fb5246da0639ba81 ...
* Retrieving assembly with id 9356b596fb5246da0639ba81
+ Found DOF: whell_back (revolute) 
+ Found DOF: whell_front (revolute) 
* Found total 2 degrees of freedom
* Found 1 root nodes:
  - BASE <1>

+ Adding part HC_L <1>
+ Adding part Part 1 <1>
+ Adding part Part 1 <2>
+ Adding part HC_R <1>
+ Adding part BASE <1>
+ Adding part HC_C <1>
+ Adding part Part 1 <3>
+ Adding part Part 1 <4>
* Writing Robot.xml
* scene.xml already exists, not over-writing it

Modelo generado exitosamente con la configuración enriquecida.


## 5. Carga y simulación interactiva en MuJoCo

Se carga el modelo generado (`HexaBot.xml` o `scene.xml`).  
Se inicializan los actuadores con los valores por defecto (las ganancias `kp`, `kv`, `forcerange` ya están en el XML).  
Se implementa un visor robusto que maneja el cierre correcto incluso en versiones antiguas de MuJoCo.

In [ ]:
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mujoco", "-q"])
    print("MuJoCo instalado correctamente.")
except subprocess.CalledProcessError as e:
    print(f"Error instalando MuJoCo: {e}")
    sys.exit(1)

from importlib.metadata import version
print(f"Versión de MuJoCo: {version('mujoco')}")

In [33]:
import mujoco
import mujoco.viewer
import signal
import sys

MODEL_XML = "model/scene.xml"  # Se usa este porque el HexaBot.xml solo contiene el robot,
                                # y el scene.xml es el que se genera con el suelo.

try:
    model = mujoco.MjModel.from_xml_path(MODEL_XML) #type: ignore
    data = mujoco.MjData(model) #type: ignore
    print(f"Modelo cargado desde {MODEL_XML}")
    print(f"   Número de articulaciones: {model.njnt}")
    print(f"   Número de actuadores: {model.nu}")
except Exception as e:
    print(f"No se pudo cargar el modelo: {e}")
    sys.exit(1)

# Inicializar controles
for i in range(model.nu):
    data.ctrl[i] = 0.0

viewer = None

def signal_handler(sig, frame):
    print("\nCerrando visor...")
    if viewer is not None:
        viewer.close()
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)

try:
    # Intento con gestor de contexto
    with mujoco.viewer.launch(model, data) as viewer: #type: ignore
        print("Visor interactivo abierto. Presiona Ctrl+C o cierra la ventana.")
        while viewer.is_running():
            mujoco.mj_step(model, data) #type: ignore
            viewer.sync()
except AttributeError:
    # Fallback a launch_passive
    print("Usando modo pasivo (launch_passive).")
    viewer = mujoco.viewer.launch_passive(model, data)
    while viewer.is_running():
        mujoco.mj_step(model, data) #type: ignore
        viewer.sync()
    viewer.close()
except Exception as e:
    print(f"Error: {e}")

Modelo cargado desde model/scene.xml
   Número de articulaciones: 3
   Número de actuadores: 2
Usando modo pasivo (launch_passive).


Generador de laberintos

In [ ]:
import xml.etree.ElementTree as ET
import trimesh
import numpy as np
import os

# Configuración
LABERT_XML = "model/labert.xml"          # Archivo original
OUTPUT_XML = "model/labert_boxes.xml"    # Archivo de salida con colisiones en cajas
ASSET_DIR = "model/assets"               # Carpeta donde están los .obj (ajústalo)

# Colores por defecto para las cajas (opcional, puedes mantener los originales)
DEFAULT_RGBA = "0.5 0.5 0.5 1"

def mesh_to_box(mesh_path):
    """Carga un archivo .obj y devuelve (center, size) en coordenadas locales."""
    if not os.path.exists(mesh_path):
        raise FileNotFoundError(f"No se encuentra {mesh_path}")
    mesh = trimesh.load(mesh_path, force='mesh')
    if mesh.is_empty:
        raise ValueError(f"El mesh {mesh_path} está vacío")
    # Bounding box axis-aligned
    min_bound = mesh.bounds[0]  # [xmin, ymin, zmin]
    max_bound = mesh.bounds[1]  # [xmax, ymax, zmax]
    center = (min_bound + max_bound) / 2.0
    size = max_bound - min_bound  # dimensiones totales
    # En MuJoCo, size = half-extents, así que dividimos entre 2
    half_size = size / 2.0
    return center, half_size

def main():
    # Cargar el XML original
    tree = ET.parse(LABERT_XML)
    root = tree.getroot()
    
    # Buscar el body que contiene los geoms (debe ser <body name="labert">)
    body = root.find(".//body[@name='labert']")
    if body is None:
        print("No se encontró el body 'labert'. Usando el primer body.")
        body = root.find("worldbody/body")
    
    # Lista de todos los geom de clase "collision"
    collision_geoms = body.findall("geom[@class='collision']")
    print(f"Encontrados {len(collision_geoms)} geoms de colisión.")
    
    # Iterar sobre cada geom de colisión
    for geom in collision_geoms:
        mesh_file = geom.get("mesh")
        if not mesh_file:
            print("Advertencia: geom sin atributo 'mesh', se omite.")
            continue
        
        # Construir la ruta completa al archivo .obj
        mesh_path = os.path.join(ASSET_DIR, mesh_file + ".obj")  # asumiendo extensión .obj
        if not os.path.exists(mesh_path):
            # Intentar con .stl? El XML original tiene .obj en las referencias (labert_collision_0.obj)
            print(f"No se encontró {mesh_path}, se omite este geom.")
            continue
        
        try:
            center, half_size = mesh_to_box(mesh_path)
        except Exception as e:
            print(f"Error procesando {mesh_file}: {e}")
            continue
        
        # Modificar el geom: cambiar type a "box", eliminar atributo "mesh", agregar "size" y "pos"
        geom.set("type", "box")
        # Eliminar el atributo "mesh" si existe
        if "mesh" in geom.attrib:
            del geom.attrib["mesh"]
        # Asignar size (half-extents) y pos (centro)
        geom.set("size", f"{half_size[0]:.6f} {half_size[1]:.6f} {half_size[2]:.6f}")
        geom.set("pos", f"{center[0]:.6f} {center[1]:.6f} {center[2]:.6f}")
        # Asegurar que tenga un color (si no tiene rgba, asignar default)
        if "rgba" not in geom.attrib:
            geom.set("rgba", DEFAULT_RGBA)
        print(f"Convertido {mesh_file} -> box size={half_size} pos={center}")
    
    # Guardar el nuevo XML
    tree.write(OUTPUT_XML, encoding="utf-8", xml_declaration=True)
    print(f"\nArchivo generado: {OUTPUT_XML}")

if __name__ == "__main__":
    main()

Encontrados 80 geoms de colisión.
Convertido labert_collision_0 -> box size=[0.13711145 4.89702725 0.12838323] pos=[2.76660064 1.9680019  0.075     ]
Convertido labert_collision_1 -> box size=[0.09688361 0.34713413 0.12515706] pos=[-1.18841081 -1.48136298  0.075     ]
Convertido labert_collision_2 -> box size=[2.38994946 0.23680824 0.12828161] pos=[0.29608042 6.62822091 0.075     ]
Convertido labert_collision_3 -> box size=[0.50033003 0.30290444 0.12709736] pos=[ 1.60677879 -0.55092314  0.075     ]
Convertido labert_collision_4 -> box size=[0.19965985 1.53137906 0.12719281] pos=[0.41971536 0.37171941 0.075     ]
Convertido labert_collision_5 -> box size=[1.20965657 0.15238122 0.12813854] pos=[0.72196419 2.05603249 0.075     ]
Convertido labert_collision_6 -> box size=[1.01091275 0.23843903 0.12838323] pos=[-1.60688753  2.74963895  0.075     ]
Convertido labert_collision_7 -> box size=[0.12634401 1.15331806 0.1237938 ] pos=[-2.74414436  2.49117484  0.075     ]
Convertido labert_collisio